# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source

The dataset's metadata and structure are defined using a Croissant schema, accessible via the URL below.

In [ ]:
# Install `mlcroissant` if necessary
!pip install mlcroissant

## 1. Data Loading

We load metadata and records from the dataset using `mlcroissant`. The Croissant schema provides structural information, including all available record sets and fields.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(url)
meta = dataset.metadata  # metadata is an object, not a dictionary or list

# Print basic metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}\nLicense: {meta.license}")

## 2. Data Overview

Explore available record sets, their fields, and IDs. With Croissant, every record set, field, and column has a unique `@id` for unambiguous reference.

In [ ]:
# List all record sets in the metadata, with their @id, name, and fields
record_sets = []
if hasattr(meta, 'record_sets'):
    # Newer versions: iterable
    for rs in meta.record_sets:
        record_sets.append(rs)
else:
    # Sometimes Croissant parses recordSet as underscore
    record_sets = getattr(meta, 'record_set', [])

print(f"Found {len(record_sets)} record set(s).")
for rs in record_sets:
    print(f"Record Set @id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description if hasattr(rs, 'description') else ''}")
    if hasattr(rs, 'fields'):
        print("  Fields (by @id):")
        for field in rs.fields:
            # Each field
            print(f"    - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', None)})")
    if hasattr(rs, 'columns'):
        print("  Columns (by @id):")
        for col in rs.columns:
            print(f"    - {col.id} (name: {col.name}, type: {getattr(col, 'data_type', None)})")
    print("")
# Keep list of their @id for later use
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. We refer to record sets and fields using their `@id` exclusively for clarity and reproducibility.

In [ ]:
# Load data from each record set @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id}: {len(df)} rows, columns: {list(df.columns)}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For exploration, use the first available record set
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nHead of record set {main_record_set_id}:")
    display(main_df.head())
else:
    # Fallback to any loaded DataFrame
    if dataframes:
        main_record_set_id, main_df = next(iter(dataframes.items()))
        print(f"\nHead of record set {main_record_set_id}:")
        display(main_df.head())
    else:
        print("No record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)

We select a numeric field (by `@id`) for basic filtering, normalization, and grouping. For demonstration, we'll assume the dataset includes GAD-7 and PHQ-9 total scores, as well as PHQ-9 depression severity and demographic fields.


In [ ]:
# Example numeric field: '@id' for GAD-7 total score (replace with actual @id if obtained from previous cell)
numeric_field_id = None
group_field_id = None

# Try to infer likely field/column IDs for GAD-7 and a demographic
available_cols = main_df.columns.tolist()
for col in available_cols:
    if 'gad' in col.lower() and 'score' in col.lower():
        numeric_field_id = col
    if (group_field_id is None) and ('gender' in col.lower() or 'sex' in col.lower()):
        group_field_id = col
if not numeric_field_id:
    # Try with PHQ or PSQ
    for col in available_cols:
        if ('phq' in col.lower() or 'psq' in col.lower()) and 'score' in col.lower():
            numeric_field_id = col

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Grouping by field @id: {group_field_id if group_field_id else 'None'}")

if numeric_field_id:
    # Filter examples: scores > threshold
    threshold = main_df[numeric_field_id].quantile(0.75)
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"\nFiltered rows where {numeric_field_id} > {threshold} (top quartile): {len(filtered_df)} samples")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst few normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by demographics (e.g., gender) and show average
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id} for filtered rows:")
        display(grouped)
else:
    print("Could not automatically select a numeric field for analysis.")

## 5. Visualization

Visualize the distribution of the selected numeric score and its relation to demographic groups, using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion

In this notebook, we demonstrated loading and exploring a FAIR² mental health survey dataset using Croissant. We referenced all entities by their `@id` and performed EDA and visualization on mental health indicators, grouped by demographic attributes if available.

Key findings and next steps depend on deeper analysis of the scores and comparison across groups, which you can further tailor to your research goals.